In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# RotationFailure-Diagnostic-V1：一次真实预检

由用户手动执行。先选择 **A100 GPU**，在 Colab Secrets 中设置 `HF_TOKEN`、`CEG_WM_ROOT_KEY` 并允许本 notebook 访问；HF 账户需已有 SD3.5 medium 模型访问权限。

固定运行提交 `b08a569`；1 张固定合成 RGB 图、未攻击和旋转 2 条记录；`science_denominator=0`。不运行 100 对诊断、不生成/嵌入图片、不调整阈值。

模型加载与预检最长 30 分钟，无自动重试。输出固定为 `/content/drive/MyDrive/CEG-WM/RotationFailure-Diagnostic-V1/preflight-v1`；已存在则停止，不覆盖。

本 notebook 已作结构与代码静态检查，未在 Colab 执行。配置依赖后如发生环境错误，保留输出并请求审计，不自行重试或换输出目录。


## 1. 检查算力、读取 Secrets、准备固定源码与依赖


In [ ]:
from pathlib import Path
from google.colab import userdata
import json
import os
import subprocess
import sys
import torch

PRODUCER_EXACT = 'b08a569eb4d456c1b391ca9e3cdee2092728c6da'
CHECKOUT = Path('/content/ceg-wm-rotation-preflight-b08a569')
RUNTIME_ROOT = Path('/content/rotation-diagnostic-runtime')
OUTPUT = Path('/content/drive/MyDrive/CEG-WM/RotationFailure-Diagnostic-V1/preflight-v1')

if OUTPUT.exists():
    raise FileExistsError('preflight-v1 already exists; preserve it and audit before another attempt')
if not torch.cuda.is_available():
    raise RuntimeError('Select an A100 GPU runtime, then run from the first cell')
gpu_name = torch.cuda.get_device_name(0)
if 'A100' not in gpu_name:
    raise RuntimeError('This frozen preflight scope requires a single A100 GPU runtime')
print('GPU:', gpu_name)
for name in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
    try:
        value = userdata.get(name)
    except Exception:
        raise RuntimeError(f'Add {name} in Colab Secrets and enable notebook access') from None
    if not value:
        raise RuntimeError(f'Empty Colab Secret: {name}')
    os.environ[name] = value
del value

if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', '--branch', 'RotationFailure-Diagnostic-V1',
                    '--single-branch', 'https://github.com/RICHAAARC/CEG-WM.git',
                    str(CHECKOUT)], check=True)
else:
    status = subprocess.check_output(['git', '-C', str(CHECKOUT), 'status', '--porcelain'], text=True)
    if status.strip():
        raise RuntimeError('Existing checkout has changes; do not overwrite them')
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', PRODUCER_EXACT], check=True)
assert subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip() == PRODUCER_EXACT
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(CHECKOUT)], check=True)
assert not subprocess.check_output(['git', '-C', str(CHECKOUT), 'status', '--porcelain'], text=True).strip()
print('Setup complete. No model has been loaded; the next cell executes the one preflight.')


## 2. 执行一次预检并读取结果

运行本单元将加载真实模型并使用 GPU。`PREFLIGHT_PASSED` 仅表示合成输入上的运行链路通过，不是诊断结论或论文结果。失败/超时后保留 `preflight-v1`，不要重复运行。


In [ ]:
if OUTPUT.exists():
    raise FileExistsError('Existing preflight-v1 is retained; do not rerun')
env = os.environ.copy()
env['ROTATION_PREFLIGHT_RUNTIME_ROOT'] = str(RUNTIME_ROOT)
env['ROTATION_PREFLIGHT_OUTPUT'] = str(OUTPUT)
env['PYTHONDONTWRITEBYTECODE'] = '1'
# The shell entry imposes a 30-minute cap including model loading, with no retry.
process = subprocess.run(
    ['bash', str(CHECKOUT / 'diagnostics/rotation_failure_v1/run_preflight.sh'),
     '--execute-authorized-preflight'],
    cwd=CHECKOUT, env=env,
)
result_path = OUTPUT / 'result.json'
if result_path.exists():
    result = json.loads(result_path.read_text())
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print('No terminal result: inspect retained started.json/condition rows. This is NOT a pass.')
if process.returncode != 0:
    raise RuntimeError(f'Preflight stopped (exit {process.returncode}); retain all files and request an audit. No automatic retry.')
